In [ ]:
# Cell 1 — Imports and data load
# Run 01_explore_clean.ipynb first: it cleans the data, makes the split and writes these files.
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                             recall_score, precision_score, f1_score,
                             roc_auc_score, accuracy_score)
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt

df = pd.read_csv("cleaned_data.csv")
with open("feature_config.json") as f:
    feature_config_in = json.load(f)

print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['at_risk'].value_counts(normalize=True)}")
print(f"\nNulls remaining: {int(df.isnull().sum().sum())}")

In [ ]:
# Cell 2 — Select model features
# Cleaning (imputation with training-split medians, one-hot encoding) happens in 01_explore_clean.ipynb.
# The model uses the 11 features listed in feature_config.json["model_features"].
FEATURES = feature_config_in['model_features']
X = df[FEATURES]
y = df['at_risk']
print(f"Feature matrix shape: {X.shape}")
print(f"Features: {FEATURES}")

In [ ]:
# Cell 3 — Train/test split (stratified 80/20, made in 01_explore_clean.ipynb)
is_train = df['split'] == 'train'
X_train, X_test = X[is_train], X[~is_train]
y_train, y_test = y[is_train], y[~is_train]
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Train at_risk rate: {y_train.mean():.3f}")
print(f"Test at_risk rate: {y_test.mean():.3f}")

In [ ]:
# Cell 4 — Baseline (majority class)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("=== BASELINE (Majority Class) ===")
print(f"Accuracy: {accuracy_score(y_test, baseline_pred):.4f}")
print(f"Recall (at_risk=1): {recall_score(y_test, baseline_pred):.4f}")
print(f"F1 (at_risk=1): {f1_score(y_test, baseline_pred):.4f}")
print(confusion_matrix(y_test, baseline_pred))
# Expected: ~72% accuracy, 0% recall — this is the bar you must clear

In [ ]:
# Cell 5 — Train decision tree with cross-validation to pick max_depth
depths = [2, 3, 4, 5, 6, 8, None]
cv_results = []

for d in depths:
    clf = DecisionTreeClassifier(
        max_depth=d,
        class_weight='balanced',
        random_state=42,
        min_samples_leaf=5  # prevent overfitting on small leaves
    )
    scores = cross_val_score(clf, X_train, y_train,
                             cv=StratifiedKFold(5), scoring='recall')
    cv_results.append((d, scores.mean(), scores.std()))
    print(f"max_depth={d}: recall={scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
# Cell 6 — Train final model (pick depth from CV results)
# Based on typical results, depth 4-5 usually balances recall and explainability
BEST_DEPTH = 4  # adjust based on cell 5 output

model = DecisionTreeClassifier(
    max_depth=BEST_DEPTH,
    class_weight='balanced',
    random_state=42,
    min_samples_leaf=5
)
model.fit(X_train, y_train)

# Training accuracy (for overfitting check)
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Overfitting gap: {train_acc - test_acc:.4f}")
# If gap > 0.10, consider reducing depth

In [ ]:
# Cell 7 — Evaluation on held-out test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== FINAL MODEL — TEST SET ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (at_risk=1):    {recall_score(y_test, y_pred):.4f}")
print(f"Precision (at_risk=1): {precision_score(y_test, y_pred):.4f}")
print(f"F1 (at_risk=1):        {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not At Risk', 'At Risk']))

In [ ]:
# Cell 8 — Feature importances
importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print("=== FEATURE IMPORTANCES ===")
print(importances.head(15))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
importances.head(12).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importance (Gini)')
ax.set_title('Top 12 Feature Importances — Decision Tree')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150)
plt.show()

In [ ]:
# Cell 9 — Export for serving
# Save model
joblib.dump(model, '../ml-service/models/tree_v1.joblib')

# Export feature list (order matters for serving)
feature_config = {
    'features': list(X.columns),
    'best_depth': BEST_DEPTH,
    'train_accuracy': float(train_acc),
    'test_accuracy': float(test_acc),
    'recall_at_risk': float(recall_score(y_test, y_pred)),
    'precision_at_risk': float(precision_score(y_test, y_pred)),
    'roc_auc': float(roc_auc_score(y_test, y_proba)),
    'feature_importances': importances.head(10).to_dict(),
    'imputation_values': feature_config_in['imputation_values']
}
with open('../ml-service/models/model_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print("Exported: ml-service/models/tree_v1.joblib + model_config.json")